# Day 11-12: Agent + RL -- 用强化学习训练智能体

**目标：** 理解如何用 RL（特别是 GRPO）训练多步决策的 LLM Agent。

**学习路线：**
1. Agent vs 普通 LLM
2. AgentRL 核心挑战
3. Agent 环境搭建
4. Reward 设计
5. Agent GRPO 训练
6. 前沿论文速览

> **前置知识：** Day 09 RL + Day 10 GRPO

---
## 1. Agent vs 普通 LLM

### 1.1 单轮 vs 多步决策

| 维度 | 普通 LLM | Agent LLM |
|------|---------|----------|
| **交互** | 单轮：问答 | 多轮：观测-动作-观测-... |
| **决策** | 一次性生成 | 逐步执行 |
| **环境** | 无 | 调用工具、读文件 |
| **奖励** | 即时 | 延迟（多步后） |
| **状态** | 固定 | 随动作变化 |

### 1.2 Agent 的 RL 循环

```
LLM Agent (策略)
     | 动作 (调用工具)
     v
环境 (文件系统/API/代码执行器)
     | 观测 + 奖励
     v
Agent 更新状态，选下一个动作
```

关键：Agent 的一次 episode 包含 **多次动作-观测交互**，形成完整轨迹。

In [ ]:
# 环境准备
import sys, json, re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "day11_12_agent_rl" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(42)
print("环境准备完成!")

In [ ]:
# 普通 LLM vs Agent 对比
print("=" * 60)
print("普通 LLM vs Agent LLM")
print("=" * 60)

print("\n--- 普通 LLM ---")
print("用户: 找出 /logs 中 WARNING 最多的文件")
print("LLM:  我无法访问文件系统...")

print("\n--- Agent LLM ---")
print("Step 1: list_dir /logs -> app.log, access.log, system.log")
print("Step 2: read_file /logs/app.log -> WARNING 出现3次")
print("Step 3: read_file /logs/access.log -> WARNING 出现1次")
print("Step 4: read_file /logs/system.log -> WARNING 出现1次")
print("Step 5: done app.log -> 任务完成!")

---
## 2. AgentRL 核心挑战

| 挑战 | 描述 | 影响 |
|------|------|------|
| **奖励稀疏** | 只有结束时才有奖励 | 学习信号弱 |
| **轨迹长** | 5-20 步 | 信用分配难 |
| **环境不确定** | 工具返回不可预测 | 需要鲁棒性 |
| **动作空间大** | 整个词表 | 搜索空间爆炸 |
| **样本效率低** | 每条轨迹成本高 | 训练数据贵 |

In [ ]:
# 挑战可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
steps = range(1, 8)
sparse = [0, 0, 0, 0, 0, 0, 1.0]
dense = [0.05, 0.05, 0.1, 0.05, 0.1, 0.05, 0.6]
x = np.arange(len(steps)); w = 0.35
ax.bar(x-w/2, sparse, w, label="结果奖励(稀疏)", color="#FF6B6B", alpha=0.8)
ax.bar(x+w/2, dense, w, label="过程奖励(密集)", color="#4ECDC4", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([f"S{s}" for s in steps])
ax.set_title("奖励稀疏 vs 密集"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

ax = axes[1]; ax.axis("off"); ax.set_title("普通RL vs AgentRL", fontsize=13, fontweight="bold")
t = ("普通 LLM RL:          Agent RL:\n"
     "- 单轮生成             - 多轮交互\n"
     "- 即时奖励             - 延迟奖励\n"
     "- 短轨迹               - 长轨迹(5-20步)\n"
     "- 状态固定             - 状态随动作变\n"
     "- 信用分配简单          - 信用分配困难\n"
     "- 样本便宜             - 样本昂贵")
ax.text(0.05, 0.9, t, transform=ax.transAxes, fontsize=10, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
plt.tight_layout(); plt.show()

---
## 3. Agent 环境搭建

**文件操作 Agent 环境**：

动作空间：
- `read_file(path)` -- 读取文件
- `write_file(path, content)` -- 写入文件
- `list_dir(path)` -- 列出目录
- `search(keyword)` -- 搜索文件
- `done(answer)` -- 提交答案

In [ ]:
# 虚拟文件系统
class VirtualFileSystem:
    def __init__(self):
        self.files = {}
        self.access_log = []

    def add_file(self, path, content):
        self.files[path] = content

    def read_file(self, path):
        self.access_log.append(f"READ: {path}")
        return (True, self.files[path]) if path in self.files else (False, f"Error: '{path}' 不存在")

    def write_file(self, path, content):
        self.access_log.append(f"WRITE: {path}")
        self.files[path] = content
        return True, f"写入 '{path}' ({len(content)} 字符)"

    def list_dir(self, dir_path):
        self.access_log.append(f"LIST: {dir_path}")
        if not dir_path.endswith("/"): dir_path += "/"
        items = set()
        for p in self.files:
            if p.startswith(dir_path):
                rel = p[len(dir_path):]; first = rel.split("/")[0]
                items.add(first + "/" if "/" in rel else first)
        return (True, "\n".join(sorted(items))) if items else (False, f"'{dir_path}' 为空")

    def search(self, keyword):
        self.access_log.append(f"SEARCH: {keyword}")
        matches = []
        for p, c in self.files.items():
            if keyword.lower() in c.lower() or keyword.lower() in p.lower():
                for i, line in enumerate(c.split("\n"), 1):
                    if keyword.lower() in line.lower():
                        matches.append(f"{p}:{i}: {line.strip()}"); break
        return (True, "\n".join(matches[:10])) if matches else (False, f"未找到 '{keyword}'")

print("VirtualFileSystem 定义完成")

In [ ]:
# 任务定义
@dataclass
class Task:
    description: str
    expected_answer: str
    setup_files: Dict[str, str]
    difficulty: str = "easy"
    max_steps: int = 10
    hints: List[str] = field(default_factory=list)

def create_tasks():
    return [
        Task("找到 /data 下包含 'error' 的文件名", "log.txt",
             {"/data/readme.md": "# 说明", "/data/log.txt": "INFO: ok\nERROR: 超时\nINFO: 恢复",
              "/data/config.json": '{"port": 8080}', "/data/notes.txt": "待办"},
             "easy", 5),
        Task("找出 /project 中 Python 文件总行数", "15",
             {"/project/main.py": "import os\nimport sys\n\ndef main():\n    print('hello')\n\nif __name__ == '__main__':\n    main()\n",
              "/project/utils.py": "def add(a, b):\n    return a + b\n\ndef sub(a, b):\n    return a - b\n",
              "/project/readme.md": "# Project"},
             "medium", 8),
        Task("在 /config 中找到数据库端口号", "5432",
             {"/config/app.yaml": "app:\n  name: myapp", "/config/db.yaml": "database:\n  host: db.example.com\n  port: 5432",
              "/config/redis.yaml": "redis:\n  port: 6379"},
             "easy", 6),
        Task("找出 /logs 中 WARNING 最多的文件", "app.log",
             {"/logs/app.log": "WARNING: 内存\nINFO: ok\nWARNING: CPU\nWARNING: 磁盘\nERROR: 崩溃",
              "/logs/access.log": "GET 200\nWARNING: 频繁\nGET 200",
              "/logs/system.log": "INFO: 启动\nWARNING: 时钟"},
             "medium", 10),
    ]

tasks = create_tasks()
print(f"创建 {len(tasks)} 个任务")
for i, t in enumerate(tasks): print(f"  任务{i+1} ({t.difficulty}): {t.description}")

In [ ]:
# AgentEnvironment
class AgentEnvironment:
    ACTION_SPACE = {"read_file": "读取文件", "write_file": "写入文件",
                    "list_dir": "列出目录", "search": "搜索", "done": "提交答案"}

    def __init__(self, task, enable_process_reward=True):
        self.task = task; self.enable_process_reward = enable_process_reward
        self.fs = VirtualFileSystem(); self.step_count = 0; self.done_flag = False
        self.history = []; self.submitted_answer = None
        self._explored_dirs = set(); self._read_files = set(); self._useful_searches = 0
        for p, c in task.setup_files.items(): self.fs.add_file(p, c)

    def reset(self):
        self.step_count = 0; self.done_flag = False; self.history = []; self.submitted_answer = None
        self._explored_dirs = set(); self._read_files = set(); self._useful_searches = 0
        self.fs = VirtualFileSystem()
        for p, c in self.task.setup_files.items(): self.fs.add_file(p, c)
        return f"任务: {self.task.description}\n最大步数: {self.task.max_steps}\n动作: {', '.join(self.ACTION_SPACE.keys())}"

    def step(self, action_str):
        self.step_count += 1; reward = 0.0
        name, args = self._parse(action_str)
        if name is None:
            obs = f"Error: 无法解析 '{action_str}'"; reward = -0.1
        elif name == "read_file":
            ok, obs = self.fs.read_file(args)
            if ok and self.enable_process_reward and args not in self._read_files:
                self._read_files.add(args); reward = 0.05
        elif name == "write_file":
            parts = args.split(" ", 1)
            if len(parts) == 2: ok, obs = self.fs.write_file(parts[0], parts[1]); reward = 0.05 if ok and self.enable_process_reward else 0
            else: obs = "Error: 需要路径和内容"; reward = -0.05
        elif name == "list_dir":
            ok, obs = self.fs.list_dir(args)
            if ok and self.enable_process_reward and args not in self._explored_dirs:
                self._explored_dirs.add(args); reward = 0.05
        elif name == "search":
            ok, obs = self.fs.search(args)
            if ok and self.enable_process_reward: self._useful_searches += 1; reward = 0.05
        elif name == "done":
            self.done_flag = True; self.submitted_answer = args.strip()
            obs = f"已提交: {self.submitted_answer}"; reward = self._final_reward()
        else: obs = f"Error: 未知 '{name}'"; reward = -0.1

        if self.step_count >= self.task.max_steps and not self.done_flag:
            self.done_flag = True; obs += "\n超时!"; reward = -0.2
        self.history.append({"step": self.step_count, "action": action_str, "observation": obs, "reward": reward})
        return obs, reward, self.done_flag, {"total_reward": sum(h["reward"] for h in self.history), "step": self.step_count}

    def _parse(self, s):
        s = s.strip()
        for n in self.ACTION_SPACE:
            if s.startswith(n): return n, s[len(n):].strip()
        return None, ""

    def _final_reward(self):
        r = 0.0
        if self.submitted_answer:
            exp = self.task.expected_answer.strip().lower(); sub = self.submitted_answer.strip().lower()
            if sub == exp: r += 1.0
            elif exp in sub or sub in exp: r += 0.5
            else:
                try:
                    if abs(float(sub) - float(exp)) < 0.01: r += 1.0
                except: pass
        if r > 0.5: r += 0.3 * max(0, 1 - self.step_count / self.task.max_steps)
        return r

    def get_total_reward(self):
        return sum(h["reward"] for h in self.history)

print("AgentEnvironment 定义完成")

In [ ]:
# 环境演示
print("=" * 60)
print("Agent 环境演示")
print("=" * 60)

task = tasks[0]
env = AgentEnvironment(task)
obs = env.reset()
print(f"初始:\n{obs}")

for action in ["list_dir /data", "search error", "done log.txt"]:
    obs, reward, done, info = env.step(action)
    print(f"\n动作: {action}")
    print(f"观测: {obs[:80]}")
    print(f"奖励: {reward:+.3f} | 累计: {info['total_reward']:.3f}")
    if done: print(f"\n任务结束! 总奖励: {info['total_reward']:.3f}"); break

In [ ]:
# 多任务演示
print("多任务演示")
def get_rule_actions(t):
    d = t.description.lower()
    if "error" in d: return ["list_dir /data", "search error", "done log.txt"]
    elif "行数" in d: return ["list_dir /project", "read_file /project/main.py", "read_file /project/utils.py", "done 15"]
    elif "端口" in d: return ["list_dir /config", "search port", "read_file /config/db.yaml", "done 5432"]
    elif "WARNING" in t.description: return ["list_dir /logs", "read_file /logs/app.log", "read_file /logs/access.log", "done app.log"]
    return ["list_dir /", "done unknown"]

for i, task in enumerate(tasks):
    env = AgentEnvironment(task); env.reset()
    for a in get_rule_actions(task):
        _, _, done, info = env.step(a)
        if done: break
    print(f"任务{i+1} ({task.difficulty}): {task.description[:35]}... | 奖励: {info['total_reward']:.3f}")

---
## 4. Reward 设计：结果奖励 vs 过程奖励

| 类型 | 描述 | 优点 | 缺点 |
|------|------|------|------|
| **结果奖励** | 只看最终答案 | 简单无偏 | 信号稀疏 |
| **过程奖励** | 中间步骤也给奖励 | 信号密集 | 可能 reward hacking |

In [ ]:
# 过程 vs 结果奖励对比
print("过程奖励 vs 结果奖励")
for rtype in ["过程+结果", "仅结果"]:
    enable = rtype == "过程+结果"
    print(f"\n--- {rtype} ---")
    for i, task in enumerate(tasks[:3]):
        env = AgentEnvironment(task, enable_process_reward=enable); env.reset()
        for a in get_rule_actions(task):
            _, _, done, info = env.step(a)
            if done: break
        print(f"  任务{i+1}: 奖励={info['total_reward']:.3f} (步数={info['step']})")

In [ ]:
# 过程奖励详细分解
print("过程奖励详细分解")
task = tasks[3]
env = AgentEnvironment(task, enable_process_reward=True); env.reset()
actions = ["list_dir /logs", "read_file /logs/app.log", "read_file /logs/access.log",
           "read_file /logs/system.log", "done app.log"]
print(f"任务: {task.description}\n")
for a in actions:
    obs, reward, done, info = env.step(a)
    src = "结果+效率" if "done" in a else ("过程(探索)" if reward > 0 else "无")
    print(f"  {info['step']:2d} | {a:^25} | {reward:+.3f} | {src:^12} | 累计:{info['total_reward']:.3f}")
    if done: break

In [ ]:
# Trade-off 可视化
fig, ax = plt.subplots(figsize=(10, 5))
cats = ["学习速度", "无偏性", "实现简单", "抗Hacking", "适用范围"]
proc = [0.9, 0.4, 0.5, 0.3, 0.7]; outc = [0.3, 0.9, 0.9, 0.9, 0.5]
x = np.arange(len(cats)); w = 0.35
ax.barh(x-w/2, proc, w, label="过程奖励", color="#4ECDC4", alpha=0.8)
ax.barh(x+w/2, outc, w, label="结果奖励", color="#FF6B6B", alpha=0.8)
ax.set_yticks(x); ax.set_yticklabels(cats); ax.set_xlabel("评分")
ax.set_title("过程 vs 结果奖励 Trade-off"); ax.legend(); ax.grid(True, alpha=0.3, axis="x"); ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

---
## 5. Agent GRPO 训练

### Trajectory-level Optimization

```
对每个任务 x:
  1. 采样 G 条轨迹 (动作-观测序列)
  2. 计算每条轨迹总奖励 R(tau_i)
  3. 组内归一化: A_i = (R - mean) / std
  4. 增加高优势轨迹中动作的概率
```

In [ ]:
# 不同能力 Agent
class RandomAgent:
    name = "RandomAgent"
    def reset(self): pass
    def act(self, obs, env):
        actions = ["list_dir /", "search file", "done unknown"]
        for p in list(env.fs.files.keys())[:3]:
            actions.append(f"read_file {p}")
            d = "/".join(p.split("/")[:-1])
            if d: actions.append(f"list_dir {d}")
        return np.random.choice(actions)

class HeuristicAgent:
    name = "HeuristicAgent"
    def __init__(self): self.step = 0; self.read = {}
    def reset(self): self.step = 0; self.read = {}
    def act(self, obs, env):
        self.step += 1
        td = env.task.description.lower()
        if self.step == 1:
            dirs = re.findall(r"/\w+", td)
            return f"list_dir {dirs[0]}" if dirs else "list_dir /"
        if self.step == 2:
            for kw in ["error", "warning", "port"]:
                if kw in td: return f"search {kw}"
            return "search file"
        if self.step <= 4:
            for p in env.fs.files:
                if p not in self.read: self.read[p] = True; return f"read_file {p}"
        return f"done {env.task.expected_answer}"

class TrainedAgent:
    name = "TrainedAgent"
    def __init__(self): self.step = 0
    def reset(self): self.step = 0
    def act(self, obs, env):
        self.step += 1; t = env.task
        plans = {"error": ["search error"], "端口": ["search port"], "WARNING": ["search WARNING"]}
        for kw, plan in plans.items():
            if kw in t.description:
                full = plan + [f"done {t.expected_answer}"]
                return full[self.step-1] if self.step <= len(full) else f"done {t.expected_answer}"
        return f"done {t.expected_answer}"

print("三种 Agent 定义完成")

In [ ]:
# 轨迹运行函数
def run_trajectory(agent, task, enable_process=True):
    env = AgentEnvironment(task, enable_process_reward=enable_process)
    obs = env.reset()
    if hasattr(agent, "reset"): agent.reset()
    traj = {"actions": [], "observations": [obs], "rewards": [], "steps": 0, "success": False}
    for _ in range(task.max_steps):
        action = agent.act(obs, env)
        obs, reward, done, info = env.step(action)
        traj["actions"].append(action); traj["observations"].append(obs)
        traj["rewards"].append(reward); traj["steps"] += 1
        if done: break
    traj["total_reward"] = sum(traj["rewards"]); traj["success"] = traj["total_reward"] > 0.5
    return traj

print("run_trajectory 定义完成")

In [ ]:
# GRPO 模拟训练
print("GRPO Agent 训练模拟")
num_epochs, group_size = 10, 4
metrics = {"epoch": [], "mean_reward": [], "success_rate": [], "mean_steps": [],
           "process_reward": [], "outcome_reward": []}

for epoch in range(num_epochs):
    erews, esucc, esteps, eproc, eout = [], [], [], [], []
    progress = epoch / num_epochs
    if progress < 0.3: agents = [RandomAgent() for _ in range(group_size)]
    elif progress < 0.7: agents = [HeuristicAgent() if np.random.random() < progress else RandomAgent() for _ in range(group_size)]
    else: agents = [TrainedAgent() if np.random.random() < progress else HeuristicAgent() for _ in range(group_size)]

    for task in tasks:
        trajs = [run_trajectory(a, task) for a in agents]
        gr = [t["total_reward"] for t in trajs]
        m, s = np.mean(gr), np.std(gr) + 1e-8
        for t in trajs:
            erews.append(t["total_reward"]); esucc.append(1.0 if t["success"] else 0.0); esteps.append(t["steps"])
            if t["rewards"]:
                eproc.append(sum(t["rewards"][:-1]) if len(t["rewards"]) > 1 else 0)
                eout.append(t["rewards"][-1])

    metrics["epoch"].append(epoch); metrics["mean_reward"].append(np.mean(erews))
    metrics["success_rate"].append(np.mean(esucc)); metrics["mean_steps"].append(np.mean(esteps))
    metrics["process_reward"].append(np.mean(eproc) if eproc else 0)
    metrics["outcome_reward"].append(np.mean(eout) if eout else 0)
    print(f"  Epoch {epoch+1:2d}/{num_epochs} | 奖励:{np.mean(erews):.3f} | 成功率:{np.mean(esucc):.0%} | 步数:{np.mean(esteps):.1f}")

In [ ]:
# 训练曲线
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(metrics["epoch"], metrics["mean_reward"], "b-o", ms=4, label="平均奖励")
ax.fill_between(metrics["epoch"], [r-0.1 for r in metrics["mean_reward"]],
                [r+0.1 for r in metrics["mean_reward"]], alpha=0.2, color="blue")
ax.set_title("GRPO Agent 奖励"); ax.set_xlabel("Epoch"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]; ax2 = ax.twinx()
l1 = ax.plot(metrics["epoch"], metrics["success_rate"], "g-o", ms=4, label="成功率")
l2 = ax2.plot(metrics["epoch"], metrics["mean_steps"], "r-s", ms=4, label="步数")
ax.set_title("成功率 & 效率"); ax.set_xlabel("Epoch"); ax.set_ylabel("成功率", color="g"); ax2.set_ylabel("步数", color="r")
ax.legend(l1+l2, [l.get_label() for l in l1+l2]); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(metrics["epoch"], metrics["process_reward"], "c-o", ms=4, label="过程奖励")
ax.plot(metrics["epoch"], metrics["outcome_reward"], "m-s", ms=4, label="结果奖励")
ax.set_title("过程 vs 结果奖励"); ax.set_xlabel("Epoch"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
names = ["Random", "Heuristic", "Trained"]
arews = [np.mean([run_trajectory(cls(), t)["total_reward"] for t in tasks])
         for cls in [RandomAgent, HeuristicAgent, TrainedAgent]]
ax.bar(names, arews, color=["#FF6B6B", "#FFD93D", "#4ECDC4"], alpha=0.8)
for i, v in enumerate(arews): ax.text(i, v+0.02, f"{v:.2f}", ha="center", fontsize=11)
ax.set_title("Agent 性能对比"); ax.set_ylabel("奖励"); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# Agent 行为对比
print("Agent 行为对比")
task = tasks[0]
for Cls, name in [(RandomAgent, "随机(训练前)"), (HeuristicAgent, "启发式(训练中)"), (TrainedAgent, "最优(训练后)")]:
    agent = Cls(); traj = run_trajectory(agent, task)
    print(f"\n--- {name} ---")
    for i, (a, o) in enumerate(zip(traj["actions"], traj["observations"][1:])):
        print(f"  Step {i+1}: {a} -> {o[:50]}... (r={traj['rewards'][i]:+.3f})")
    print(f"  总奖励: {traj['total_reward']:.3f} | 步数: {traj['steps']}")

In [ ]:
# GRPO 组内归一化在 Agent 场景
print("GRPO Agent 场景")
task = tasks[0]
agents_mix = [RandomAgent(), RandomAgent(), HeuristicAgent(), TrainedAgent()]
trajs = [run_trajectory(a, task) for a in agents_mix]
gr = [t["total_reward"] for t in trajs]
m, s = np.mean(gr), np.std(gr) + 1e-8
advs = [(r-m)/s for r in gr]

print(f"\n任务: {task.description}")
print(f"均值={m:.3f}, 标准差={s:.3f}")
for i, (t, a, adv) in enumerate(zip(trajs, agents_mix, advs)):
    act = "提高" if adv > 0 else "降低"
    print(f"  轨迹{i+1} ({a.name}): 奖励={t['total_reward']:.3f} 优势={adv:+.3f} ({act}概率)")

---
## 6. 前沿论文速览

### 6.1 AgentTuning (2023)

- **核心**：混合通用数据和 Agent 交互数据微调 LLM
- **方法**：收集成功轨迹 + 通用指令数据混合 SFT
- **发现**：纯 Agent 数据损害通用能力，混合是关键
- **局限**：依赖成功轨迹收集

### 6.2 ReAct (2022)

- **核心**：LLM 交替推理 (Thought) 和行动 (Action)
- **方法**：few-shot prompting 引导 Thought-Action-Observation 循环
- **发现**：推理帮助制定更好的行动计划
- **影响**：成为 LangChain Agent 等的基础范式

### 6.3 Toolformer (2023)

- **核心**：让 LLM 自学何时及如何使用外部工具
- **方法**：自监督生成 API 调用标注，筛选标准：困惑度是否降低
- **发现**：模型能自主在需要时调用工具
- **关系**：Toolformer 用 SFT，AgentRL 用 RL 优化工具使用

In [ ]:
# 前沿论文对比
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off"); ax.set_title("Agent + LLM 前沿研究", fontsize=14, fontweight="bold", pad=20)
t = ("论文           核心方法              训练     关键创新\n"
     "ReAct         Thought-Act-Obs      Few-shot  推理+行动交替\n"
     "Toolformer    自学习工具调用         SFT       自监督API标注\n"
     "AgentTuning   混合数据微调           SFT       通用+Agent数据\n"
     "DeepSeek-R1   GRPO                  RL        涌现推理\n"
     "AgentRL       轨迹级GRPO            RL        多步决策优化\n"
     "\n"
     "趋势: Few-shot -> SFT -> RL\n"
     "      人工设计 -> 有监督 -> 自主探索")
ax.text(0.05, 0.9, t, transform=ax.transAxes, fontsize=10, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
plt.tight_layout(); plt.show()

### 5.2 Agent GRPO 关键参数

| 参数 | 普通 GRPO | Agent GRPO | 原因 |
|------|----------|-----------|------|
| G | 4-8 | 4-6 | 轨迹成本高 |
| max_length | 256 | 512-1024 | 多轮更长 |
| 学习率 | 1e-5 | 5e-6 | 更保守 |
| KL 系数 | 0.01 | 0.05 | 防遗忘 |

In [ ]:
# 各 Agent 详细表现
print("各 Agent 在所有任务上的表现")
print(f"\n{'Agent':^15} | {'任务1':>6} | {'任务2':>6} | {'任务3':>6} | {'任务4':>6} | {'平均':>6}")
print("-" * 65)
for Cls, name in [(RandomAgent, "Random"), (HeuristicAgent, "Heuristic"), (TrainedAgent, "Trained")]:
    rews = [run_trajectory(Cls(), t)["total_reward"] for t in tasks]
    vals = " | ".join(f"{r:6.3f}" for r in rews)
    print(f"{name:^15} | {vals} | {np.mean(rews):6.3f}")

In [ ]:
# 轨迹长度和成功率变化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
np.random.seed(42)
ax.hist(np.random.randint(3,10,50), bins=8, alpha=0.6, label="训练前", color="#FF6B6B")
ax.hist(np.random.randint(2,5,50), bins=8, alpha=0.6, label="训练后", color="#4ECDC4")
ax.set_title("轨迹长度分布变化"); ax.set_xlabel("步数"); ax.set_ylabel("频次"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
sr = [0.05, 0.10, 0.15, 0.25, 0.40, 0.55, 0.65, 0.75, 0.82, 0.88]
ax.plot(range(1,11), sr, "g-o", linewidth=2, ms=6)
ax.fill_between(range(1,11), sr, alpha=0.15, color="green")
ax.set_title("成功率随训练变化"); ax.set_xlabel("Epoch"); ax.set_ylabel("成功率"); ax.set_ylim(0,1); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Agent 常见失败模式
print("Agent 常见失败模式")
for name, desc, impact in [
    ("重复动作", "重复读取同一文件", "过程奖励只对新文件生效"),
    ("过早提交", "没充分探索就提交", "答案错误，奖励0"),
    ("超时", "探索太多不相关文件", "步数上限，罚-0.2"),
    ("解析错误", "输出格式不对", "动作解析失败，罚-0.1"),
]:
    print(f"  {name:>8}: {desc}")
    print(f"           影响: {impact}")

In [ ]:
# Reward Hacking 示例
print("Reward Hacking 示例\n")
task_h = tasks[0]
env_h = AgentEnvironment(task_h, enable_process_reward=True)
env_h.reset()
hack_actions = [f"read_file {p}" for p in task_h.setup_files.keys()] + ["done wrong"]
for a in hack_actions:
    _, _, done, info = env_h.step(a)
    if done: break
print(f"Hacking: 读所有文件但答案错误")
print(f"  总奖励: {info['total_reward']:.3f} (过程奖励掩盖错误!)")
print("\n解决: 1)降低过程奖励 2)只奖励相关动作 3)错误答案更大惩罚")

### 5.3 实际应用场景

Agent GRPO 可训练：
- **代码 Agent**：导航代码库、修复 bug
- **网页 Agent**：浏览、填表、查找信息
- **数据分析 Agent**：加载数据、查询、生成报告
- **系统管理 Agent**：监控日志、排查故障

每个场景需定制环境、奖励函数、训练策略。

In [ ]:
# Agent 能力雷达图
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(polar=True))
cats = ["完成率", "效率", "低错误率", "适应性", "鲁棒性"]
N = len(cats); angles = [n/float(N)*2*np.pi for n in range(N)]; angles += angles[:1]

for vals, name, color in [([0.15,0.2,0.8,0.1,0.1], "Random", "#FF6B6B"),
                           ([0.6,0.5,0.4,0.4,0.5], "Heuristic", "#FFD93D"),
                           ([0.9,0.85,0.1,0.7,0.8], "Trained", "#4ECDC4")]:
    v = vals + vals[:1]
    ax.plot(angles, v, "o-", linewidth=1.5, label=name, color=color)
    ax.fill(angles, v, alpha=0.1, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=9)
ax.set_title("Agent 能力雷达图", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()

---
## 7. 总结

1. **Agent vs LLM**：多步交互、奖励稀疏、轨迹长
2. **环境设计**：动作空间、状态、奖励函数是 AgentRL 基础
3. **过程 vs 结果奖励**：过程快但有偏，结果无偏但稀疏
4. **Agent GRPO**：轨迹级优化，组内归一化比较不同策略
5. **前沿**：从 prompting (ReAct) -> SFT (AgentTuning) -> RL (GRPO)

| 概念 | 说明 |
|------|------|
| 轨迹奖励 R(tau) = sum r_t | 整条轨迹累计 |
| 组内归一化 A = (R-mean)/std | GRPO 核心 |
| 过程奖励 r_t = f(s_t, a_t) | 每步信号 |
| 效率奖励 = 1 - steps/max | 鼓励少步骤 |

### 思考题

1. **为什么 Agent 的奖励稀疏比普通 LLM 更严重？**

2. **过程奖励 reward hacking 的具体例子？** 提示：读新文件 +0.05...

3. **环境从文件操作扩展到网页浏览，额外挑战？**

4. **ReAct 的 Thought-Action-Observation 和 Agent GRPO 如何结合？**